# 01 · Auditoría, preparación y análisis exploratorio
**¿Qué se midió y qué falta?** Cada fila es un cultivo; cada columna, una señal del laboratorio. Este cuaderno trabaja solo con mediciones originales. El [cuaderno 02](02_imputacion_evaluacion.ipynb) evalúa los métodos y estudia los efectos de las estimaciones.

Desde la raíz del repositorio, con el entorno de **requirements.txt**, se puede ejecutar de arriba abajo. Cambie `METABOLITO` para consultar cualquier columna seleccionada; el nombre corto se encuentra en el inventario.

In [ ]:
import pandas as pd
from protocolo.datos import PRIV, PUB, leer_matriz, preparar
from protocolo.distribuciones import describir
from protocolo.figuras import por_metabolito_original

from IPython.display import Image, display
from protocolo.figuras import FIG

def mostrar_figura(nombre):
    display(Image(filename=str(FIG / (nombre + ".png"))))

METABOLITO = "fum"
preparar()
ishii = leer_matriz("ishii")
from protocolo.figuras import original
if not (FIG / "01_mediciones_y_ausencias.png").exists():
    original(ishii)
inventario = pd.read_csv(PUB / "inventario.csv")
print(f"{len(ishii)} cultivos, {len(ishii.columns)} metabolitos, {ishii.count().sum()} mediciones y {ishii.isna().sum().sum()} huecos.")

## De la base experimental a la matriz
Ishii aporta 30 cultivos principales en glucosa a una dilución de 0,2 h⁻¹. Las cuatro condiciones WT a otras diluciones son un análisis de sensibilidad. Las 135 señales tienen identidad defendible dentro de Ishii y al menos diez mediciones en el inventario inicial. Dentro de cada entrenamiento se vuelve a comprobar la disponibilidad. Una sustancia puede ser útil en Ishii aunque no tenga equivalencia demostrada en Gerosa.

La [base SQLite en mM](data/concentraciones_mM.db) conserva las mediciones y los blancos. Ishii ya informa mM. Para Gerosa se divide µmol/g de peso seco por 2,3 mL/g, obteniendo µmol/mL = mM. No se divide por la concentración de Gerosa en glucosa. MMC2 aporta ocho condiciones auxiliares y no aporta verdades de evaluación.

In [ ]:
ejemplos = ["fum", "pep", "s7p", "succ", "cit", "atp", "ish_g6p", "ish_f6p", "ish_ru5p", "ish_mal", "dhap", "nadh", "nadph"]
display(inventario.loc[inventario.variable.isin(ejemplos), [
    "variable", "original_label", "protocol", "compartment", "identidad", "n_30", "decision"
]].set_index("variable"))

## ¿Cuánta información hay antes de imputar?
El porcentaje de ausencias y la dispersión tienen significados distintos. La desviación estándar (SD) está en mM; el CV expresa SD/media en porcentaje y se omite si la media es casi cero. La asimetría solo se calcula con ocho observaciones y dispersión suficiente. Los extremos de las cajas se revisan en la fuente y no se descartan.

In [ ]:
# Cobertura, variabilidad, ausencias y ejemplos de mediciones
mostrar_figura("01_mediciones_y_ausencias")

In [ ]:
resumen = describir(ishii)
display(resumen.reindex([METABOLITO, "cit", "atp"]).drop_duplicates().round(4))
# Consulta completa: resumen o salida_protocolo/revision/resultados/estadistica_original.csv

In [ ]:
import matplotlib.pyplot as plt
por_metabolito_original(METABOLITO, ishii)
plt.show()

## La identidad y la celda de origen
«Atípico» significa que una concentración queda fuera de Q1 − 1,5 IQR o Q3 + 1,5 IQR. Eso puede reflejar la respuesta del cultivo o una particularidad de medición. La celda y el protocolo permiten revisarla; el criterio estadístico no distingue ambas explicaciones.

In [ ]:
traza = pd.read_csv(PRIV / "trazabilidad.csv.gz", low_memory=False)
ejemplo = traza[traza.variable.eq(METABOLITO) & traza.sample_key.isin(ishii.index)]
display(ejemplo[["sample_key", "variable", "source_sheet", "source_cell", "protocol", "valor_mM", "absence_code"]].head(6))

La asociación se calcula por parejas con al menos ocho cultivos que tengan ambas mediciones. Las correlaciones originales están en la primera columna de la figura, tanto para Pearson como para Spearman. El [cuaderno 02](02_imputacion_evaluacion.ipynb) explica por qué pueden cambiar cuando se agregan estimaciones.

In [ ]:
pares = ishii[["fum", "pep", "s7p", "succ"]]
display(pares.corr(method="spearman", min_periods=8).round(2))
disponible = pares.notna().astype(int)
display(disponible.T @ disponible)